# Organoid Regulome Benchmark: hgx on Fleck et al. 2023
GPU-accelerated hypergraph deep learning on cerebral organoid gene regulatory networks.
Requires Colab Pro/Pro+ with **A100 GPU** runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/organoid-hgx-benchmark'
DATA_DIR = f'{DRIVE_DIR}/data'
os.makedirs(f'{DRIVE_DIR}/figures', exist_ok=True)

In [ ]:
# Install deps + clone repos (pulls latest fixes)
!pip install -q jax[cuda12] equinox diffrax optax jaxtyping scanpy anndata ripser matplotlib networkx

# Clone or update hgx
if os.path.exists('/content/hgx'):
    !cd /content/hgx && git pull
else:
    !git clone https://github.com/m9h/hgx.git /content/hgx
!pip install -q -e /content/hgx

# Clone or update devograph (dynamics, GRN, perturbation, topology extensions)
if os.path.exists('/content/devograph'):
    !cd /content/devograph && git pull
else:
    !git clone https://github.com/m9h/devograph.git /content/devograph
!pip install -q -e /content/devograph

# Clone or update benchmark
if os.path.exists('/content/organoid-hgx-benchmark'):
    !cd /content/organoid-hgx-benchmark && git pull
else:
    !git clone https://github.com/m9h/organoid-hgx-benchmark.git /content/organoid-hgx-benchmark

# Verify
import jax
print(f'JAX {jax.__version__} on {jax.devices()}')
import hgx, devograph
print(f'hgx OK, devograph: from_snapshots={hasattr(devograph, "from_snapshots")}, '
      f'PerturbationPredictor={hasattr(devograph, "PerturbationPredictor")}, '
      f'HypergraphNeuralODE={hasattr(devograph, "HypergraphNeuralODE")}')

In [ ]:
# Download Zenodo data to Drive (skips if already cached)
files = {
    'pando/coefs.tsv': 'https://zenodo.org/records/15371701/files/grn_modules.tsv?download=1',
    'expression/RNA_data.h5ad': 'https://zenodo.org/records/15371701/files/RNA_data.h5ad?download=1',
    'zenodo/rna_matrices.tar.gz': 'https://zenodo.org/records/15371701/files/rna_matrices.tar.gz?download=1',
    'zenodo/RNA_all_velo.h5ad': 'https://zenodo.org/records/15371701/files/RNA_all_velo.h5ad?download=1',
    'zenodo/motifs.tar.gz': 'https://zenodo.org/records/15371701/files/motifs.tar.gz?download=1',
}
for path, url in files.items():
    fp = f'{DATA_DIR}/{path}'
    os.makedirs(os.path.dirname(fp), exist_ok=True)
    if os.path.exists(fp):
        print(f'  CACHED: {path} ({os.path.getsize(fp)/1e6:.0f} MB)')
    else:
        print(f'  Downloading {path}...')
        subprocess.run(['curl', '-L', '-o', fp, url], check=True)
        print(f'    Done: {os.path.getsize(fp)/1e6:.0f} MB')

# Extract count matrices
meta_dir = f'{DATA_DIR}/zenodo/data_matrices'
if not os.path.exists(meta_dir):
    !tar xzf {DATA_DIR}/zenodo/rna_matrices.tar.gz -C {DATA_DIR}/zenodo/
print('All data ready!')

In [ ]:
# Symlink data + figures to Drive
work_dir = '/content/organoid-hgx-benchmark'
for name, target in [('data', DATA_DIR), ('figures', f'{DRIVE_DIR}/figures')]:
    link = f'{work_dir}/{name}'
    if os.path.islink(link): os.unlink(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    os.symlink(target, link)

# Preprocess (PPCA dimensionality estimation)
!cd {work_dir} && python scripts/00_preprocess.py

In [ ]:
# Run full benchmark
!cd /content/organoid-hgx-benchmark && python scripts/run_all_real.py --epochs 200

In [ ]:
# Display all figures
from IPython.display import display, Image
from pathlib import Path
for f in sorted(Path(f'{DRIVE_DIR}/figures').glob('*.png')):
    print(f'\n{"="*60}\n  {f.name}\n{"="*60}')
    display(Image(filename=str(f), width=900))